
# Narrative Similarity — Track B (Single-Story Embeddings) 📘

This notebook builds **Track B** embeddings for the Narrative Similarity task using **only each story itself at inference** (Track-B compliant).  
It supports:
- **Aspect-aware text construction** from a single story (Theme / Action / Outcome).
- **(Optional) Fine-tuning** a Sentence-Transformers model on training triples (anchor, A, B).
- **Export** to `track_b.jsonl` and `track_b.npy` in dataset order.
- **Dev evaluation**: accuracy on dev triples.

> **Inputs expected** (adjust paths if needed):
> - `data/train_triples.jsonl` (optional, for training)  
> - `data/dev_triples.jsonl` (optional, for evaluation)  
> - `data/items.jsonl` (required for Track B export; one story per line with key `text`)


In [1]:

# %%capture
# If needed, uncomment to install deps
# !pip install -U sentence-transformers tqdm numpy pandas scikit-learn


In [ ]:
# %% [setup]
# !pip install -U sentence-transformers tqdm numpy pandas scikit-learn

import os, json, re, math, random
from pathlib import Path
from typing import List, Dict, Any, Tuple
import numpy as np
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)


BASE_MODEL = "BAAI/bge-large-en-v1.5"

TRAIN_TRIPLES = Path("train_triples.jsonl")   # optional but recommended
DEV_TRIPLES   = Path("dev_triples.jsonl")     # optional eval
ITEMS_FILE    = Path("dev_track_b.jsonl")     # your items (one line per story, key 'text')

OUT_JSONL = Path("track_b.jsonl")
OUT_NPY   = Path("track_b.npy")
OUT_ZIP   = Path("codabench_track_b.zip")
MODEL_OUT = Path("models/trackB_aspect_bge_large")  # where to save fine-tuned model
MODEL_OUT.mkdir(parents=True, exist_ok=True)


In [13]:
# %% [aspect extraction]
SENT_SPLIT_RE = re.compile(r'(?<=[.!?])\s+')

def split_sentences(text: str) -> List[str]:
    text = (text or "").strip()
    if not text: return []
    return [s.strip() for s in SENT_SPLIT_RE.split(text) if s.strip()]

def pick_theme(text: str, max_sents: int = 3) -> str:
    s = split_sentences(text);  return " ".join(s[:max_sents]) if s else text

def pick_outcome(text: str, max_sents: int = 2) -> str:
    s = split_sentences(text);  return " ".join(s[-max_sents:]) if s else text

ACTION_VERB_CUES = {
    "go","goes","went","travel","travels","traveled","reach","reaches","reached",
    "fight","fights","fought","escape","escapes","escaped","search","searches","searched",
    "find","finds","found","discover","discovers","discovered","return","returns","returned",
    "kill","kills","killed","die","dies","died","plan","plans","planned","attack","attacks","attacked",
    "try","tries","tried","start","starts","started","begin","begins","began","decide","decides","decided",
    "save","saves","saved","help","helps","helped","learn","learns","learned","investigate","investigates","investigated",
    "betray","betrays","betrayed","lose","loses","lost","win","wins","won"
}
def extract_action_chain(text: str, max_items: int = 6) -> str:
    picked, sents = [], split_sentences(text)
    for s in sents:
        toks = re.findall(r"[A-Za-z']+", s.lower())
        if any(t in ACTION_VERB_CUES or t.endswith("ed") or t.endswith("ing") for t in toks):
            picked.append(s)
        if len(picked) >= max_items: break
    if not picked and sents: picked = sents[:min(3, len(sents))]
    return " | ".join(picked)

def build_aspect_text(story: str) -> str:
    return f"THEME: {pick_theme(story)}\nACTION: {extract_action_chain(story)}\nOUTCOME: {pick_outcome(story)}"


In [14]:
# %% [io helpers]
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists(): return []
    out = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            out.append(json.loads(line))
    return out

def read_items_jsonl(path: Path) -> List[str]:
    items = []
    with path.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            txt = obj.get("text") or obj.get("story") or obj.get("content") or ""
            if not isinstance(txt, str):
                raise ValueError(f"missing 'text' on line {ln}")
            items.append(txt)
    return items


In [5]:
# %% [Export helpers]
def read_items_jsonl(path: Path) -> List[str]:
    items = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            items.append(obj.get("text") or obj.get("story") or obj.get("content") or "")
    return items

def export_track_b(items_path: Path, out_jsonl: Path, out_npy: Path) -> np.ndarray:
    assert items_path.exists(), f"Missing items file: {items_path}"
    stories = read_items_jsonl(items_path)
    assert len(stories) > 0, "No stories found."

    aspect_texts = [build_aspect_text(s) for s in stories]          # single-story → aspect string
    X = st_embed_texts(aspect_texts).astype("float32")              # embeddings

    with out_jsonl.open("w", encoding="utf-8") as f:
        for row in X.tolist():
            f.write(json.dumps({"embeddings": row}) + "\n")

    np.save(out_npy, X)
    print(f"Wrote {len(stories)} embeddings → {out_jsonl}, {out_npy}")
    return X

def zip_submission(jsonl_path: Path, npy_path: Path, zip_path: Path):
    import zipfile
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(jsonl_path, arcname="track_b.jsonl")
        z.write(npy_path,   arcname="track_b.npy")
    print(f"Created zip → {zip_path}")


In [15]:
# %% [embedder + model load]
from sentence_transformers import SentenceTransformer

# For BGE/GTE it helps to add a short instruction; STILL single-story.
DOC_PREFIX = ""
if "bge" in BASE_MODEL.lower():
    DOC_PREFIX = "Represent the passage for retrieval: "
elif "e5" in BASE_MODEL.lower():
    DOC_PREFIX = "passage: "

model = SentenceTransformer(BASE_MODEL)
print("Loaded:", BASE_MODEL)

def build_doc_text(story: str) -> str:
    return DOC_PREFIX + build_aspect_text(story)

def st_embed_texts(texts: List[str]) -> np.ndarray:
    return np.asarray(
        model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
    )


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\rishe\anaconda3\envs\rimsh\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rishe\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loaded: BAAI/bge-large-en-v1.5


In [7]:
# %% [Optional fine-tuning]
DO_TRAIN = False    # flip to True if you have train_triples.jsonl
TRAIN_TRIPLES = Path("train_triples.jsonl")
EPOCHS, BATCH, LR, WARMUP = 3, 64, 2e-5, 0.1

def build_training_pairs(triples: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    pairs = []
    for t in triples:
        anchor = t.get('anchor') or t.get('Anchor') or t.get('anchor_text') or t.get('anchorStory') or ""
        A = t.get('A') or t.get('a') or t.get('candA') or ""
        B = t.get('B') or t.get('b') or t.get('candB') or ""
        label = t.get('text_a_is_closer')
        if label is None:
            lab = t.get('label')
            if isinstance(lab, str):
                label = (lab.strip().upper() == 'A')
        if not anchor or not A or not B or label is None: 
            continue
        chosen = A if label else B
        pairs.append((build_aspect_text(anchor), build_aspect_text(chosen)))
    return pairs

def fine_tune_sentence_transformer(model, train_pairs: List[Tuple[str, str]]):
    from sentence_transformers import losses, InputExample, datasets
    train_examples = [InputExample(texts=[a, b]) for a, b in train_pairs]
    train_dl = datasets.NoDuplicatesDataLoader(train_examples, batch_size=BATCH)
    loss = losses.MultipleNegativesRankingLoss(model)
    warmup_steps = int(len(train_dl) * EPOCHS * WARMUP)
    model.fit(train_objectives=[(train_dl, loss)],
              epochs=EPOCHS, warmup_steps=warmup_steps,
              scheduler='linear', optimizer_params={'lr': LR},
              show_progress_bar=True)
    return model

if DO_TRAIN and USE_SENTENCE_TRANSFORMERS:
    triples = read_triples_jsonl(TRAIN_TRIPLES)
    pairs = build_training_pairs(triples)
    print(f"Training on {len(pairs)} pairs…")
    MODEL = fine_tune_sentence_transformer(MODEL, pairs)


In [16]:
# %% [fine-tuning]
DO_TRAIN = True               # set False to skip training
EPOCHS, BATCH, LR, WARMUP = 3, 64, 2e-5, 0.1

from sentence_transformers import losses, InputExample, datasets

def build_training_pairs(triples: List[Dict[str, Any]]) -> List[Tuple[str, str]]:
    pairs = []
    for t in triples:
        anchor = t.get('anchor') or t.get('Anchor') or t.get('anchor_text') or t.get('anchorStory') or ""
        A = t.get('A') or t.get('a') or t.get('candA') or ""
        B = t.get('B') or t.get('b') or t.get('candB') or ""
        label = t.get('text_a_is_closer')
        if label is None:
            lab = t.get('label')
            if isinstance(lab, str):
                label = (lab.strip().upper() == 'A')
        if not anchor or not A or not B or label is None: 
            continue
        chosen = A if label else B

        # aspect-augmented positives (all from single stories)
        anc_full = build_doc_text(anchor);     pos_full = build_doc_text(chosen)
        anc_th   = DOC_PREFIX + f"THEME: {pick_theme(anchor)}"
        pos_th   = DOC_PREFIX + f"THEME: {pick_theme(chosen)}"
        anc_ac   = DOC_PREFIX + f"ACTION: {extract_action_chain(anchor)}"
        pos_ac   = DOC_PREFIX + f"ACTION: {extract_action_chain(chosen)}"
        anc_out  = DOC_PREFIX + f"OUTCOME: {pick_outcome(anchor)}"
        pos_out  = DOC_PREFIX + f"OUTCOME: {pick_outcome(chosen)}"

        pairs.extend([
            (anc_full, pos_full),
            (anc_th,   pos_th),
            (anc_ac,   pos_ac),
            (anc_out,  pos_out),
        ])
    return pairs

if DO_TRAIN and TRAIN_TRIPLES.exists():
    triples = read_jsonl(TRAIN_TRIPLES)
    train_pairs = build_training_pairs(triples)
    print("Train pairs:", len(train_pairs))

    train_examples = [InputExample(texts=[a, b]) for a, b in train_pairs]
    train_dl = datasets.NoDuplicatesDataLoader(train_examples, batch_size=BATCH)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    warmup_steps = int(len(train_dl) * EPOCHS * WARMUP)
    model.fit(
        train_objectives=[(train_dl, train_loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        scheduler='linear',
        optimizer_params={'lr': LR},
        show_progress_bar=True
    )
    model.save(str(MODEL_OUT))
    print("Saved fine-tuned model to:", MODEL_OUT)
else:
    print("Skipping training (set DO_TRAIN=True and supply train_triples.jsonl).")


Skipping training (set DO_TRAIN=True and supply train_triples.jsonl).


In [17]:
# %% [dev eval]
def accuracy_on_triples(triples_path: Path) -> float:
    triples = read_jsonl(triples_path)
    if not triples:
        print("No dev triples found.")
        return 0.0
    rights = 0
    for t in tqdm(triples, desc="Evaluating"):
        anchor = t.get('anchor') or t.get('Anchor') or t.get('anchor_text') or t.get('anchorStory') or ""
        A = t.get('A') or t.get('a') or t.get('candA') or ""
        B = t.get('B') or t.get('b') or t.get('candB') or ""
        label = t.get('text_a_is_closer')
        if label is None:
            lab = t.get('label')
            if isinstance(lab, str):
                label = (lab.strip().upper() == 'A')
        if not anchor or not A or not B or label is None: 
            continue
        anc, eA, eB = st_embed_texts([build_doc_text(anchor), build_doc_text(A), build_doc_text(B)])
        pred_is_A = float(np.dot(anc, eA)) > float(np.dot(anc, eB))
        rights += int(pred_is_A == bool(label))
    acc = rights / len(triples)
    print(f"Dev accuracy: {acc:.4f} on {len(triples)} items")
    return acc

if DEV_TRIPLES.exists():
    _ = accuracy_on_triples(DEV_TRIPLES)


In [18]:
# %% [export]
def export_track_b(items_path: Path, out_jsonl: Path, out_npy: Path) -> np.ndarray:
    stories = read_items_jsonl(items_path)
    aspect_texts = [build_doc_text(s) for s in stories]     # SINGLE-STORY → aspect string
    X = st_embed_texts(aspect_texts).astype("float32")

    with out_jsonl.open("w", encoding="utf-8") as f:
        for row in X.tolist():
            f.write(json.dumps({"embeddings": row}) + "\n")
    np.save(out_npy, X)
    print(f"Wrote {len(stories)} embeddings → {out_jsonl}, {out_npy}")
    return X

X = export_track_b(ITEMS_FILE, OUT_JSONL, OUT_NPY)

# zip for CodaBench
import zipfile
with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(OUT_JSONL, arcname="track_b.jsonl")
    z.write(OUT_NPY,   arcname="track_b.npy")
print("Zipped:", OUT_ZIP, " | Embedding shape:", X.shape)


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Wrote 479 embeddings → track_b.jsonl, track_b.npy
Zipped: codabench_track_b.zip  | Embedding shape: (479, 1024)


In [19]:
# %% [nn sanity]
stories = read_items_jsonl(ITEMS_FILE)
S = X @ X.T
pairs = []
for i in range(len(stories)):
    for j in range(i+1, len(stories)):
        pairs.append((i, j, float(S[i,j])))
pairs.sort(key=lambda x: x[2], reverse=True)
for k,(i,j,s) in enumerate(pairs[:10],1):
    print(f"{k:02d}. ({i},{j}) sim={s:.4f}")


01. (67,373) sim=0.8856
02. (135,287) sim=0.8589
03. (68,333) sim=0.8554
04. (85,116) sim=0.8499
05. (101,257) sim=0.8495
06. (180,226) sim=0.8490
07. (11,373) sim=0.8446
08. (68,111) sim=0.8413
09. (262,419) sim=0.8407
10. (30,262) sim=0.8388


In [24]:
def accuracy_on_triples(triples_path: Path) -> float:
    # ... (code to load and process triples) ...
    acc = rights / len(triples)
    print(f"Dev accuracy: {acc:.4f} on {len(triples)} items")
    return acc

    print(accuracy_on_triples(Path("data/dev_triples.jsonl")))